In [ ]:
import os
import re
import gc
import json
import warnings

import numpy as np
import pandas as pd
from scipy import signal
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

CH_PATH = "ch.parquet"
KO_PATH = "ko.parquet"
OUT_DIR = "rebuild_v3"
N_JOBS = -1
SAVE_DTYPE = np.float32

FS = 500
RAW_LEN = 10 * FS
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF",
         "V1", "V2", "V3", "V4", "V5", "V6"]

COL_CANDIDATES = {
    "patient_id": ["PatientID", "Patient_ID", "patient_id", "PID"],
    "ecg_id":     ["ECG_ID", "ECGID", "ecg_id", "FileName", "filename", "ECG_File"],
    "age":        ["Age", "age", "연령", "AgeDays", "Age_days"],
    "sex":        ["Gender", "Sex", "sex", "성별"],
    "aha":        ["AHA_code", "AHA_Code", "AHA"],
    "chn":        ["CHN_code", "CHN_Code", "CHN"],
    "icd":        ["ICD-10 code", "ICD10", "ICD_10", "ICD-10"],
    "k":          ["K", "Potassium", "potassium", "K_value", "칼륨", "serum_K"],
    "lab_dt_min": ["time_diff_min", "lab_time_diff", "dt_min"],
    "y":          ["y", "label"],
}

AGE_UNIT = {"china": "auto", "korea": "auto"}
SEX_NUMERIC_FEMALE = 1

AHA_HYPER = r"\bSuggests208\b"
AHA_HYPO = r"\bSuggests211\b"
K_HYPO = 3.3
K_HYPER = 5.5

STT_AHA = ["L145", "L146", "L147", "L150"]
STT_CHN = ["L121", "L122", "L123", "L128"]
TALL_T_REGEX = r"prominent\s*T\s*wave|tall\s*T\s*wave|peaked\s*T"
TALL_T_TEXT_COLS = None

KO_STATEMENT_FLAGS = {
    "flag_poor_quality":   r"Poor data quality",
    "flag_undetermined":   r"Undetermined rhythm",
    "flag_low_voltage":    r"Low voltage QRS",
    "flag_early_repol":    r"Early repolarization",
    "flag_nonspecific_st": r"Nonspecific ST abnormality",
    "flag_prolonged_qt":   r"Prolonged QT",
    "flag_nonspecific_t":  r"Nonspecific T wave abnormality",
}

PRE = int(round(0.25 * FS))
POST = int(round(0.55 * FS))
CROP_LEN = PRE + POST
MIN_BEATS = 3
QRS_WIN = (PRE - int(round(0.06 * FS)), PRE + int(round(0.10 * FS)))
ECTOPIC_CORR = 0.80
ECTOPIC_RR_RATIO = 0.80
ECTOPIC_RR_NEIGHBORS = 3
GAUSS_SIGMA = 0.08 * CROP_LEN

POST_R_START = int(round(0.08 * FS))
POST_R_END_MAX = int(round(0.36 * FS))
POST_R_END_RR_FRAC = 0.70
CYCLE_END_RR_FRAC = 0.90
MIN_VALID_LEN = PRE + POST_R_START + 20

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def show_schema(path, n_rows=3):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(path)
    print(f"\n[{path}] rows = {pf.metadata.num_rows}")
    sample = next(pf.iter_batches(batch_size=n_rows)).to_pandas()
    for col in sample.columns:
        value = sample[col].dropna()
        kind = type(value.iloc[0]).__name__ if len(value) else "empty"
        extra = ""
        if len(value) and isinstance(value.iloc[0], (list, np.ndarray)):
            extra = f"  (array, length {len(value.iloc[0])})"
        elif len(value) and isinstance(value.iloc[0], str) and len(value.iloc[0]) > 200:
            extra = f"  (long string, length {len(value.iloc[0])})"
        print(f"  {str(col):<30} {str(sample[col].dtype):<10} {kind}{extra}")


show_schema(CH_PATH)
show_schema(KO_PATH)

In [ ]:
def pick_col(df, key, required=True):
    for name in COL_CANDIDATES[key]:
        if name in df.columns:
            return name
    if required:
        raise KeyError(f"No column found for '{key}'. Add the column name to COL_CANDIDATES['{key}'].\n"
                       f"Available columns: {list(df.columns)}")
    return None


def canonical_lead_names(df):
    upper_map = {lead.upper(): lead for lead in LEADS}
    rename = {}
    for col in df.columns:
        key = str(col).upper()
        if key in upper_map and col != upper_map[key]:
            rename[col] = upper_map[key]
    return df.rename(columns=rename)


def parse_signal(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, str):
        try:
            arr = np.asarray(json.loads(value), dtype=np.float32)
        except Exception:
            nums = re.findall(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", value)
            arr = np.asarray(nums, dtype=np.float32) if nums else None
    else:
        arr = np.asarray(value, dtype=np.float32)
    if arr is None or arr.size == 0:
        return None
    return arr.ravel()


def prepare_age(series, unit, cohort_name):
    age = pd.to_numeric(series, errors="coerce").astype(float)
    values = age.dropna()
    if unit == "auto":
        if len(values) and values.isin([0, 1, 2, 3]).all():
            unit = "category"
        elif values.max() > 200:
            unit = "days"
        else:
            unit = "years"
        print(f"  [{cohort_name}] age unit: {unit} (min {values.min():.1f}, max {values.max():.1f})")

    if unit == "category":
        print(f"  [{cohort_name}] age is stored as categories 0-3; age-based exclusion is not possible. "
              f"Put a column with age in years first in COL_CANDIDATES['age'].")
        age_cat = age.fillna(-1).astype(np.int64).to_numpy()
        return np.full(len(age), np.nan), age_cat

    age_years = (age / 365.25 if unit == "days" else age).to_numpy()
    return age_years, age_category(age_years)


def age_category(age_years):
    a = np.asarray(age_years, dtype=float)
    cat = np.full(len(a), -1, dtype=np.int64)
    cat[(a >= 1) & (a < 7)] = 0
    cat[(a >= 7) & (a < 10)] = 1
    cat[(a >= 10) & (a < 13)] = 2
    cat[a >= 13] = 3
    return cat


def encode_sex(values):
    out = np.full(len(values), -1, dtype=np.int64)
    for i, value in enumerate(values):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            continue
        text = str(value).strip().strip("'\"").strip().upper()
        if text in {"0", "1", "0.0", "1.0"}:
            out[i] = 1 if int(float(text)) == SEX_NUMERIC_FEMALE else 0
        elif text.startswith("F") or text in {"女", "여", "여자", "WOMAN"}:
            out[i] = 1
        elif text.startswith("M") or text in {"男", "남", "남자", "MAN"}:
            out[i] = 0
    n_missing = int((out < 0).sum())
    if n_missing:
        print(f"Missing or unrecognized sex: {n_missing} / {len(out)}")
    return out


def code_token_regex(codes):
    return r"(?<![A-Za-z0-9])(?:" + "|".join(map(re.escape, codes)) + r")(?![0-9])"


def text_contains(df, cols, pattern):
    if not cols:
        return np.zeros(len(df), dtype=bool)
    joined = df[cols].astype(str).agg(" ".join, axis=1)
    return joined.str.contains(pattern, case=False, regex=True).to_numpy()

In [ ]:
SOS_BP = signal.butter(4, [0.67, 40.0], btype="band", fs=FS, output="sos")
B_QRS, A_QRS = signal.butter(3, [9.0, 30.0], btype="band", fs=FS)


def detect_r_peaks(lead_ii):
    qrs = signal.filtfilt(B_QRS, A_QRS, lead_ii)
    envelope = np.abs(qrs)
    peaks, _ = signal.find_peaks(envelope, distance=int(0.30 * FS),
                                 prominence=2.0 * np.std(envelope))
    half = int(0.05 * FS)
    centered = np.abs(lead_ii - np.median(lead_ii))
    refined = []
    for p in peaks:
        start, end = max(0, p - half), min(len(lead_ii), p + half + 1)
        i = int(np.argmax(centered[start:end])) + start
        offset = 0.0
        if 0 < i < len(lead_ii) - 1:
            a, b, c = centered[i - 1], centered[i], centered[i + 1]
            denom = a - 2 * b + c
            if denom != 0:
                offset = float(np.clip(0.5 * (a - c) / denom, -0.5, 0.5))
        refined.append(i + offset)
    refined = np.sort(np.asarray(refined, dtype=float))
    kept = []
    for r in refined:
        if not kept or r - kept[-1] >= 0.20 * FS:
            kept.append(r)
    return np.asarray(kept, dtype=float)


def premature_flags(r_peaks):
    rr = np.diff(r_peaks)
    flags = np.zeros(len(r_peaks), dtype=bool)
    for k in range(1, len(r_peaks)):
        j = k - 1
        lo, hi = max(0, j - ECTOPIC_RR_NEIGHBORS), min(len(rr), j + ECTOPIC_RR_NEIGHBORS + 1)
        neighbors = np.delete(rr[lo:hi], j - lo)
        if len(neighbors) == 0:
            continue
        flags[k] = rr[j] < ECTOPIC_RR_RATIO * np.median(neighbors)
    return flags


def build_crops(x12, r_peaks):
    if len(r_peaks) < 2:
        return None, "rpeak_lt2"
    median_rr = float(np.median(np.diff(r_peaks)))
    premature_all = premature_flags(r_peaks)

    beats, premature = [], []
    for k, r in enumerate(np.round(r_peaks).astype(int)):
        if r - PRE < 0 or r + POST > x12.shape[1]:
            continue
        beats.append(x12[:, r - PRE:r + POST])
        premature.append(premature_all[k])
    if len(beats) < MIN_BEATS:
        return None, "beats_lt_min"

    beats = np.stack(beats).astype(np.float32)
    premature = np.asarray(premature, dtype=bool)

    qrs = beats[:, 1, QRS_WIN[0]:QRS_WIN[1]]
    template = np.median(qrs, axis=0)
    corr = np.array([np.corrcoef(q, template)[0, 1] for q in qrs])
    corr = np.nan_to_num(corr, nan=-1.0)
    keep = (~premature) & (corr >= ECTOPIC_CORR)
    if keep.sum() < MIN_BEATS:
        return None, "clean_beats_lt_min"

    clean = beats[keep]
    median_beat = np.median(clean, axis=0)
    p90_beat = np.percentile(clean, 90, axis=0)
    weight = np.exp(-((np.arange(CROP_LEN) - PRE) ** 2) / (2 * GAUSS_SIGMA ** 2))
    gauss_beat = weight * median_beat + (1 - weight) * p90_beat

    info = {
        "median_rr": median_rr,
        "n_beats_total": int(len(beats)),
        "n_beats_used": int(keep.sum()),
        "n_ectopic": int((~keep).sum()),
        "n_premature": int(premature.sum()),
        "n_morph_outlier": int((corr < ECTOPIC_CORR).sum()),
    }
    return {"median": median_beat, "gauss": gauss_beat}, info


def cycle_end_index(median_rr):
    return int(np.clip(PRE + round(CYCLE_END_RR_FRAC * median_rr), MIN_VALID_LEN, CROP_LEN))


def derived_features(crop, median_rr):
    start = PRE + POST_R_START
    end = PRE + min(POST_R_END_MAX, int(round(POST_R_END_RR_FRAC * median_rr)))
    end = int(np.clip(end, start + 2, CROP_LEN))
    cycle_end = max(cycle_end_index(median_rr), end)
    post = crop[:, start:end]
    energy_ratio = float(np.sum(post ** 2) / (np.sum(crop[:, :cycle_end] ** 2) + 1e-8))
    abs_slope = float(np.mean(np.abs(np.diff(post, axis=1))))
    return energy_ratio, abs_slope


def legacy_crop(x12):
    sig_len = 400
    lead_ii = x12[1]
    b, a = signal.butter(3, [9.0 / (0.5 * FS), 30.0 / (0.5 * FS)], btype="band")
    filtered = signal.filtfilt(b, a, lead_ii)
    peaks, _ = signal.find_peaks(filtered, distance=int(0.3 * FS),
                                 prominence=np.std(filtered) * 2)
    refined = []
    search = int(0.05 * FS)
    for p in peaks:
        s, e = max(0, int(p) - search), min(len(lead_ii), int(p) + search)
        if s >= e:
            continue
        i = int(np.argmax(lead_ii[s:e])) + s
        if 0 < i < len(lead_ii) - 1:
            al, be, ga = lead_ii[i - 1], lead_ii[i], lead_ii[i + 1]
            d = al - 2 * be + ga
            refined.append(i + 0.5 * (al - ga) / d if d != 0 else i)
        else:
            refined.append(i)
    refined = np.asarray(refined)
    r_crop = refined[(refined >= 500) & (refined <= 4500)]
    if len(r_crop) < 3:
        return None

    expected_r = int(sig_len * (0.33 / (0.33 + 0.60)))
    weight = np.exp(-((np.arange(sig_len) - expected_r) ** 2) / (2 * (sig_len * 0.08) ** 2))
    crops, valid_lens = [], []
    for lead_idx in range(12):
        sliced = []
        for i in range(len(r_crop) - 1):
            cur, nxt = r_crop[i], r_crop[i + 1]
            rr = nxt - cur
            s, e = int(round(cur - rr * 0.33)), int(round(cur + rr * 0.60))
            if s < 0 or e > RAW_LEN:
                continue
            beat = x12[lead_idx, s:e]
            if lead_idx == 0:
                valid_lens.append(min(len(beat), sig_len))
            beat = beat[:sig_len] if len(beat) > sig_len else np.pad(beat, (0, sig_len - len(beat)))
            sliced.append(beat)
        if not sliced:
            return None
        sliced = np.asarray(sliced, dtype=np.float32)
        crops.append((1 - weight) * np.percentile(sliced, 90, axis=0) + weight * np.median(sliced, axis=0))
    crop = np.stack(crops).astype(np.float32)
    valid_len = int(np.clip(round(float(np.median(valid_lens))), 1, sig_len))

    ps = max(0, min(expected_r + int(0.08 * FS), sig_len - 2))
    pe = max(ps + 1, min(expected_r + int(0.36 * FS), sig_len))
    post = crop[:, ps:pe]
    energy = float(np.mean(post ** 2) / (np.mean(crop ** 2) + 1e-8))
    slope = float(np.mean(np.abs(np.diff(post, axis=1))))
    hr = (60 * FS) / np.mean(np.diff(r_crop))
    return {"crop": crop, "valid_len": valid_len, "hr": hr, "energy": energy, "slope": slope}


def process_record(lead_values):
    out = {"status": "error"}
    try:
        raw = []
        for lead in LEADS:
            sig = parse_signal(lead_values.get(lead))
            if sig is None or not np.isfinite(sig).all():
                out["status"] = "missing_lead"
                return out
            if np.ptp(sig) == 0:
                out["status"] = "flat_lead"
                return out
            raw.append(sig)

        length = min(len(s) for s in raw)
        out["duration_sec"] = length / FS
        if length < RAW_LEN:
            out["status"] = "short_lt10s"
            return out

        x12 = np.stack([signal.sosfiltfilt(SOS_BP, s[:length])[:RAW_LEN] for s in raw]).astype(np.float32)

        scale = float(np.median(np.abs(x12)))
        if scale < 1e-8:
            out["status"] = "flat_signal"
            return out
        x12 = x12 / scale
        out["amp_scale"] = scale

        r_peaks = detect_r_peaks(x12[1])
        out["n_rpeaks"] = int(len(r_peaks))
        crops, info = build_crops(x12, r_peaks)
        if crops is None:
            out["status"] = info
            return out

        median_rr = info["median_rr"]
        out.update(info)
        out["hr"] = 60.0 * FS / median_rr
        out["rr_sd_ms"] = float(np.std(np.diff(r_peaks)) / FS * 1000)
        out["x_raw"] = x12.astype(SAVE_DTYPE)
        for mode, crop in crops.items():
            energy, slope = derived_features(crop, median_rr)
            out[f"crop_{mode}"] = crop.astype(SAVE_DTYPE)
            out[f"energy_{mode}"] = energy
            out[f"slope_{mode}"] = slope

        legacy = legacy_crop(x12)
        out["legacy_ok"] = legacy is not None
        if legacy is not None:
            out["crop_legacy"] = legacy["crop"].astype(SAVE_DTYPE)
            out["valid_len_legacy"] = legacy["valid_len"]
            out["hr_legacy"] = legacy["hr"]
            out["energy_legacy"] = legacy["energy"]
            out["slope_legacy"] = legacy["slope"]

        out["status"] = "ok"
        return out
    except Exception as exc:
        out["status"] = "error"
        out["error"] = f"{type(exc).__name__}: {exc}"
        return out


def run_signal_processing(df):
    records = df[LEADS].to_dict("records")
    return Parallel(n_jobs=N_JOBS, batch_size=8)(
        delayed(process_record)(r) for r in tqdm(records, desc="Processing ECGs")
    )

In [ ]:
ch = canonical_lead_names(pd.read_parquet(CH_PATH, engine="pyarrow"))
missing_leads = [lead for lead in LEADS if lead not in ch.columns]
if missing_leads:
    print(f"Lead columns not found: {missing_leads}; records are marked missing_lead")
    for lead in missing_leads:
        ch[lead] = None

c_pid = pick_col(ch, "patient_id")
c_eid = pick_col(ch, "ecg_id", required=False)
c_age = pick_col(ch, "age")
c_sex = pick_col(ch, "sex")
c_aha = pick_col(ch, "aha")
c_chn = pick_col(ch, "chn")
c_icd = pick_col(ch, "icd", required=False)

ch_meta = pd.DataFrame({
    "patient_id": ch[c_pid].astype(str).values,
    "ecg_id": ch[c_eid].astype(str).values if c_eid else np.arange(len(ch)).astype(str),
})
ch_meta["age_years"], ch_meta["age_cat"] = prepare_age(ch[c_age], AGE_UNIT["china"], "development")
ch_meta["sex"] = encode_sex(ch[c_sex].values)

aha_text = ch[c_aha].astype(str)
chn_text = ch[c_chn].astype(str)
is_hyper = aha_text.str.contains(AHA_HYPER, regex=True).to_numpy()
is_hypo = aha_text.str.contains(AHA_HYPO, regex=True).to_numpy()
ch_meta["y"] = np.where(is_hyper & ~is_hypo, 1, np.where(is_hypo & ~is_hyper, 2, 0))
ch_meta["flag_label_conflict"] = is_hyper & is_hypo
ch_meta.loc[ch_meta["flag_label_conflict"], "y"] = -1

ch_meta["flag_stt_noise"] = (
    aha_text.str.contains(code_token_regex(STT_AHA), regex=True).to_numpy()
    | chn_text.str.contains(code_token_regex(STT_CHN), regex=True).to_numpy()
)
if TALL_T_TEXT_COLS is None:
    auto_cols = [c for c in ch.columns
                 if re.search(r"statement|diag|desc|诊断|text", str(c), flags=re.IGNORECASE)]
    tall_cols = list(dict.fromkeys([c_aha, c_chn] + auto_cols))
else:
    tall_cols = TALL_T_TEXT_COLS
print(f"Columns searched for tall T statements: {tall_cols}")
ch_meta["flag_tall_t"] = text_contains(ch, tall_cols, TALL_T_REGEX)
if ch_meta["flag_tall_t"].sum() == 0:
    print("No tall T statements found; check TALL_T_TEXT_COLS and TALL_T_REGEX.")

if c_icd:
    icd_text = ch[c_icd].astype(str)
    ch_meta["icd_e876"] = icd_text.str.contains("E87.6", regex=False).to_numpy()
    ch_meta["icd_e875"] = icd_text.str.contains("E87.5", regex=False).to_numpy()

print("Development labels:", ch_meta["y"].value_counts().to_dict())
print("Tall T flagged:", int(ch_meta["flag_tall_t"].sum()), "| ST-T flagged:", int(ch_meta["flag_stt_noise"].sum()))

In [ ]:
ko = canonical_lead_names(pd.read_parquet(KO_PATH, engine="pyarrow")).reset_index(drop=True)
for lead in [lead for lead in LEADS if lead not in ko.columns]:
    ko[lead] = None

K_COL = "검사결과1"
ECG_TIME_COL = "Study_Time"
LAB_TIME_COL = "실시일시"
DT_COL = None
LAB_ID_COL = None
PAIR_WINDOW_MIN = 60
ONE_ECG_PER_LAB = True
K_DECIMALS = 1

print("External columns (leads excluded):", [c for c in ko.columns if c not in LEADS])
if DT_COL is None and (ECG_TIME_COL is None or LAB_TIME_COL is None):
    raise ValueError("Set ECG_TIME_COL and LAB_TIME_COL, or DT_COL.")

k_pid = pick_col(ko, "patient_id")
k_eid = pick_col(ko, "ecg_id", required=False)
k_age = pick_col(ko, "age")
k_sex = pick_col(ko, "sex")

ko_meta = pd.DataFrame({
    "patient_id": ko[k_pid].astype(str).values,
    "ecg_id": ko[k_eid].astype(str).values if k_eid else np.arange(len(ko)).astype(str),
})
age_raw = pd.to_numeric(ko[k_age], errors="coerce").astype(float)
if age_raw.max(skipna=True) > 200:
    age_raw = age_raw / 365.25
ko_meta["age_years"] = age_raw.values
ko_meta["age_cat"] = age_category(ko_meta["age_years"])
ko_meta["age_over14"] = ko_meta["age_years"] >= 15
ko_meta["sex"] = encode_sex(ko[k_sex].values)

k_raw = pd.to_numeric(ko[K_COL], errors="coerce")
n_not_numeric = int(k_raw.isna().sum() - ko[K_COL].isna().sum())
potassium = np.round(k_raw.to_numpy(dtype=float), K_DECIMALS)
k_valid = potassium[~np.isnan(potassium)]
print(f"Potassium '{K_COL}': min {k_valid.min():.1f}, max {k_valid.max():.1f}, "
      f"missing {int(ko[K_COL].isna().sum())}, non-numeric {n_not_numeric}")
if np.isin(k_valid, [0, 1, 2]).all():
    raise ValueError(f"'{K_COL}' contains only 0/1/2 and does not look like potassium values.")

y = np.where(potassium < K_HYPO, 2, np.where(potassium >= K_HYPER, 1, 0))
y[np.isnan(potassium)] = -1
ko_meta["k_value"] = potassium
ko_meta["y"] = y

ecg_time = pd.to_datetime(ko[ECG_TIME_COL], errors="coerce") if ECG_TIME_COL else None
lab_time = pd.to_datetime(ko[LAB_TIME_COL], errors="coerce") if LAB_TIME_COL else None
if ECG_TIME_COL:
    print(f"ECG time unparsed: {int(ecg_time.isna().sum())}")
if LAB_TIME_COL:
    print(f"Lab time unparsed: {int(lab_time.isna().sum())}")

if DT_COL is not None:
    dt_min = pd.to_numeric(ko[DT_COL], errors="coerce").to_numpy(dtype=float)
else:
    dt_min = ((lab_time - ecg_time).dt.total_seconds() / 60).to_numpy(dtype=float)
abs_dt = np.abs(dt_min)
ko_meta["lab_dt_min"] = dt_min

valid_dt = abs_dt[~np.isnan(abs_dt)]
print("\n|lab - ECG| (min):",
      pd.Series(valid_dt).describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1).to_dict())
for window in (5, 10, 30, 60, 1440):
    print(f"  <= {window} min: {int((valid_dt <= window).sum())} ({100 * (valid_dt <= window).mean():.1f}%)")
print(f"Lab before ECG {int((dt_min < 0).sum())}, after {int((dt_min > 0).sum())}, "
      f"same time {int((dt_min == 0).sum())}")

if k_eid:
    ecg_key = ko_meta["ecg_id"].to_numpy()
elif ecg_time is not None:
    ecg_key = (ko_meta["patient_id"] + "|" + ecg_time.astype(str)).to_numpy()
else:
    ecg_key = None
    print("No ECG ID or ECG time; duplicate ECGs cannot be identified.")
if LAB_ID_COL:
    lab_key = ko[LAB_ID_COL].astype(str).to_numpy()
elif lab_time is not None:
    lab_key = (ko_meta["patient_id"] + "|" + lab_time.astype(str)).to_numpy()
else:
    lab_key = None

CLASS_NAMES = {0: "normal", 1: "hyper", 2: "hypo"}
keep = np.ones(len(ko), dtype=bool)
pair_flow = []


def apply_step(name, step_mask):
    global keep
    removed = keep & ~step_mask
    keep = keep & step_mask
    row = {"step": name, "removed": int(removed.sum()), "remaining": int(keep.sum())}
    for code, cname in CLASS_NAMES.items():
        row[f"removed_{cname}"] = int((removed & (y == code)).sum())
        row[f"remaining_{cname}"] = int((keep & (y == code)).sum())
    pair_flow.append(row)


def nearest_only(key):
    candidates = pd.DataFrame({"key": key, "abs_dt": abs_dt, "row": np.arange(len(ko))})[keep]
    best = candidates.sort_values(["abs_dt", "row"]).drop_duplicates("key", keep="first")["row"].to_numpy()
    mask = np.zeros(len(ko), dtype=bool)
    mask[best] = True
    return mask


apply_step("all rows in ko.parquet", np.ones(len(ko), dtype=bool))
apply_step("potassium missing or non-numeric", ~np.isnan(potassium))
apply_step("ECG or lab time missing", ~np.isnan(dt_min))
apply_step(f"|lab - ECG| > {PAIR_WINDOW_MIN} min", abs_dt <= PAIR_WINDOW_MIN)
if ecg_key is not None:
    n_multi = int(pd.Series(ecg_key[keep]).duplicated(keep=False).sum())
    print(f"\nRows of ECGs with >1 potassium value within {PAIR_WINDOW_MIN} min: {n_multi}")
    apply_step("same ECG with >1 potassium: keep nearest", nearest_only(ecg_key))
if ONE_ECG_PER_LAB and lab_key is not None:
    apply_step("same potassium with >1 ECG: keep nearest", nearest_only(lab_key))

pair_flow = pd.DataFrame(pair_flow)
print("\n" + pair_flow.to_string(index=False))
pair_flow.to_csv(os.path.join(OUT_DIR, "korea_pairing_flow.csv"), index=False)

ko = ko.loc[keep].reset_index(drop=True)
ko_meta = ko_meta.loc[keep].reset_index(drop=True)
gc.collect()

statement_cols = sorted(
    [c for c in ko.columns if re.match(r"^statement_\d+$", str(c), flags=re.IGNORECASE)],
    key=lambda c: int(re.search(r"_(\d+)$", str(c)).group(1)),
)
print(f"\nStatement columns: {len(statement_cols)}")
for flag_name, pattern in KO_STATEMENT_FLAGS.items():
    ko_meta[flag_name] = text_contains(ko, statement_cols, pattern)
ko_meta["flag_any_statement"] = ko_meta[list(KO_STATEMENT_FLAGS)].any(axis=1)
ko_meta["flag_tall_t"] = text_contains(ko, statement_cols, TALL_T_REGEX)

print(f"After pairing: {len(ko_meta)} ECGs, {ko_meta['patient_id'].nunique()} patients")
print("External labels:", ko_meta["y"].value_counts().to_dict())

In [ ]:
ch_results = run_signal_processing(ch)
ko_results = run_signal_processing(ko)

In [ ]:
SCALAR_KEYS = ["status", "error", "duration_sec", "amp_scale", "n_rpeaks", "median_rr",
               "n_beats_total", "n_beats_used", "n_ectopic", "n_premature", "n_morph_outlier",
               "hr", "rr_sd_ms", "energy_median", "slope_median", "energy_gauss", "slope_gauss",
               "legacy_ok", "valid_len_legacy", "hr_legacy", "energy_legacy", "slope_legacy"]


def attach_results(meta, results):
    meta = meta.copy()
    for key in SCALAR_KEYS:
        meta[key] = [r.get(key, np.nan) for r in results]
    meta["legacy_ok"] = meta["legacy_ok"].fillna(False).astype(bool)
    return meta


def eligibility_steps(meta, cohort):
    status = meta["status"]
    if meta["age_years"].notna().any():
        if cohort == "china":
            age_ok = meta["age_years"] >= 1
        else:
            age_ok = (meta["age_years"] >= 1) & (meta["age_years"] < 18)
    else:
        age_ok = meta["age_cat"].between(0, 3)
    steps = [
        ("incomplete 12-lead (missing or flat lead)", ~status.isin(["missing_lead", "flat_lead"])),
        ("age <1 y or missing",                       age_ok.fillna(False)),
        ("recording <10 s",                           status != "short_lt10s"),
        ("signal/beat QC failure",                    status == "ok"),
        ("label missing or conflict",                 meta["y"] >= 0),
    ]
    return [(name, np.asarray(mask, dtype=bool)) for name, mask in steps]


def save_npz(meta, results, keep, path):
    idx = np.where(keep)[0]
    sub = meta.iloc[idx].reset_index(drop=True)
    n = len(idx)

    X_raw = np.empty((n, 12, RAW_LEN), dtype=SAVE_DTYPE)
    X_crop = np.zeros((n, 12, CROP_LEN), dtype=SAVE_DTYPE)
    X_gauss = np.zeros((n, 12, CROP_LEN), dtype=SAVE_DTYPE)
    X_legacy = np.zeros((n, 12, CROP_LEN), dtype=SAVE_DTYPE)
    for row, i in enumerate(idx):
        r = results[i]
        x = r.pop("x_raw", None)
        if x is None:
            raise RuntimeError("Raw signals were already released; rerun signal processing.")
        X_raw[row] = x
        X_crop[row] = r["crop_median"]
        X_gauss[row] = r["crop_gauss"]
        if "crop_legacy" in r:
            X_legacy[row] = r["crop_legacy"]

    base = np.stack([sub["sex"], sub["age_cat"]], axis=1).astype(np.float32)

    def meta_matrix(hr, energy, slope):
        return np.concatenate([base, np.stack([hr, energy, slope], axis=1).astype(np.float32)], axis=1)

    legacy_ok = sub["legacy_ok"].to_numpy()
    valid_len = np.array([cycle_end_index(rr) if np.isfinite(rr) else CROP_LEN
                          for rr in sub["median_rr"].to_numpy(dtype=float)], dtype=np.int64)
    arrays = {
        "X_raw": X_raw,
        "X_crop": X_crop,
        "X_crop_gauss": X_gauss,
        "X_crop_legacy": X_legacy,
        "crop_valid_len": valid_len,
        "crop_valid_len_legacy": sub["valid_len_legacy"].fillna(CROP_LEN).astype(np.int64).to_numpy(),
        "M": meta_matrix(sub["hr"], sub["energy_median"], sub["slope_median"]),
        "M_gauss": meta_matrix(sub["hr"], sub["energy_gauss"], sub["slope_gauss"]),
        "M_legacy": meta_matrix(sub["hr_legacy"].fillna(sub["hr"]),
                                sub["energy_legacy"].fillna(0), sub["slope_legacy"].fillna(0)),
        "legacy_ok": legacy_ok,
        "y": sub["y"].astype(np.int64).to_numpy(),
        "ids": sub["patient_id"].astype(str).to_numpy(),
        "ecg_ids": sub["ecg_id"].astype(str).to_numpy(),
        "age_years": sub["age_years"].astype(np.float32).to_numpy(),
        "amp_scale": sub["amp_scale"].astype(np.float32).to_numpy(),
        "n_beats_used": sub["n_beats_used"].astype(np.int64).to_numpy(),
        "n_ectopic": sub["n_ectopic"].astype(np.int64).to_numpy(),
    }
    for col in sub.columns:
        if col.startswith("flag_") or col.startswith("icd_") or col == "age_over14":
            arrays[col] = sub[col].fillna(False).astype(bool).to_numpy()
    for col in ("k_value", "lab_dt_min"):
        if col in sub.columns:
            arrays[col] = sub[col].astype(np.float64).to_numpy()

    np.savez_compressed(path, **arrays)
    print(f"Saved {path}: {n} ECGs, {sub['patient_id'].nunique()} patients, "
          f"legacy crop failed {int((~legacy_ok).sum())}")
    print(f"  crop_valid_len: median {np.median(valid_len):.0f}, < 400 samples "
          f"{100 * np.mean(valid_len < CROP_LEN):.1f}%, min {valid_len.min()}")
    del X_raw, X_crop, X_gauss, X_legacy, arrays
    gc.collect()
    return sub


ch_meta = attach_results(ch_meta, ch_results)
ko_meta = attach_results(ko_meta, ko_results)

ch_steps = eligibility_steps(ch_meta, "china")
ko_steps = eligibility_steps(ko_meta, "korea")
ch_keep = np.logical_and.reduce([m for _, m in ch_steps])
ko_keep = np.logical_and.reduce([m for _, m in ko_steps])

ch_meta.to_csv(os.path.join(OUT_DIR, "china_records.csv"), index=False)
ko_meta.to_csv(os.path.join(OUT_DIR, "korea_records.csv"), index=False)
ch_final = save_npz(ch_meta, ch_results, ch_keep, os.path.join(OUT_DIR, "china_v2.npz"))
ko_final = save_npz(ko_meta, ko_results, ko_keep, os.path.join(OUT_DIR, "korea_v2.npz"))

In [ ]:
CLASS_NAMES = {0: "normal", 1: "hyper", 2: "hypo"}


def count_row(meta, mask, label):
    mask = np.asarray(mask, dtype=bool)
    y = meta["y"].to_numpy()
    row = {"step": label, "ECG_total": int(mask.sum()),
           "patients_total": int(meta.loc[mask, "patient_id"].nunique())}
    for code, name in CLASS_NAMES.items():
        m = mask & (y == code)
        row[f"ECG_{name}"] = int(m.sum())
        row[f"patients_{name}"] = int(meta.loc[m, "patient_id"].nunique())
    return row


def flow_table(meta, steps):
    keep = np.ones(len(meta), dtype=bool)
    rows = [count_row(meta, keep, "start")]
    for name, mask in steps:
        rows.append(count_row(meta, keep & ~mask, f"excluded: {name}"))
        keep &= mask
        rows.append(count_row(meta, keep, f"remaining after: {name}"))
    return pd.DataFrame(rows)


def flags_by_class(final):
    rows = []
    for col in [c for c in final.columns if c.startswith("flag_") or c.startswith("icd_") or c == "age_over14"]:
        rows.append(count_row(final, final[col].fillna(False).astype(bool).to_numpy(), col))
    return pd.DataFrame(rows)


ch_flow = flow_table(ch_meta, ch_steps)
ko_flow = flow_table(ko_meta, ko_steps)
ch_flow.to_csv(os.path.join(OUT_DIR, "china_flow.csv"), index=False)
ko_flow.to_csv(os.path.join(OUT_DIR, "korea_flow.csv"), index=False)
flags_by_class(ch_final).to_csv(os.path.join(OUT_DIR, "china_flags_by_class.csv"), index=False)
flags_by_class(ko_final).to_csv(os.path.join(OUT_DIR, "korea_flags_by_class.csv"), index=False)

with open(os.path.join(OUT_DIR, "china_label_check.txt"), "w", encoding="utf-8") as f:
    chn_hypo = chn_text.str.contains("hypokalemia", case=False, regex=False).to_numpy()
    chn_hyper = chn_text.str.contains("hyperkalemia", case=False, regex=False).to_numpy()
    f.write("[AHA vs CHN, hypokalemia, all records]\n"
            + pd.crosstab(is_hypo, chn_hypo, rownames=["AHA"], colnames=["CHN"]).to_string() + "\n\n")
    f.write("[AHA vs CHN, hyperkalemia, all records]\n"
            + pd.crosstab(is_hyper, chn_hyper, rownames=["AHA"], colnames=["CHN"]).to_string() + "\n\n")
    f.write(f"[ECGs with both codes] {int(ch_meta['flag_label_conflict'].sum())}\n\n")
    if "icd_e876" in ch_final.columns:
        f.write("[Final cohort, ICD-10 E87.6 vs label, ECG level]\n"
                + pd.crosstab(ch_final["icd_e876"], ch_final["y"]).to_string() + "\n\n")
        f.write("[Final cohort, ICD-10 E87.5 vs label, ECG level]\n"
                + pd.crosstab(ch_final["icd_e875"], ch_final["y"]).to_string() + "\n\n")
        patient = ch_final.groupby("patient_id").agg(
            hypo_label=("y", lambda s: bool((s == 2).any())), e876=("icd_e876", "max"))
        f.write("[Final cohort, ICD-10 E87.6 vs hypokalemia label, patient level]\n"
                + pd.crosstab(patient["e876"], patient["hypo_label"]).to_string() + "\n")

print(ch_flow.to_string(index=False))
print(ko_flow.to_string(index=False))
print("\nStatus, development:", ch_meta["status"].value_counts().to_dict())
print("Status, external:", ko_meta["status"].value_counts().to_dict())
ok = ch_meta["status"] == "ok"
print(f"\nExcluded beats per ECG, development: {ch_meta.loc[ok, 'n_ectopic'].mean():.2f} "
      f"(premature {ch_meta.loc[ok, 'n_premature'].mean():.2f}, morphology {ch_meta.loc[ok, 'n_morph_outlier'].mean():.2f})")

In [ ]:
def load_v2(path):
    return dict(np.load(path, allow_pickle=True))


def cohort_mask(data, exclude_flags=()):
    mask = np.ones(len(data["y"]), dtype=bool)
    for flag in exclude_flags:
        mask &= ~data[flag].astype(bool)
    return mask


def patient_level_label(ids, y, positive=2):
    frame = pd.DataFrame({"id": ids, "pos": (np.asarray(y) == positive)})
    grouped = frame.groupby("id")["pos"].max()
    return grouped.index.to_numpy(), grouped.to_numpy().astype(int)


def fit_meta_scaler(M, train_mask, cols=(2, 3, 4)):
    from sklearn.preprocessing import RobustScaler
    return RobustScaler().fit(M[train_mask][:, list(cols)])


def apply_meta_scaler(M, scaler, cols=(2, 3, 4)):
    scaled = M.copy()
    scaled[:, list(cols)] = scaler.transform(M[:, list(cols)])
    return scaled